<a href="https://colab.research.google.com/github/Mansik-04/assignments/blob/main/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
    "Q1. Role of filters & feature maps in CNN":
    "Filters (kernels) extract spatial features by sliding across the image. "
    "Feature maps are the outputs after applying filters, representing learned features "
    "like edges, textures, or objects.",

    "Q2. Padding & Stride":
    "Padding preserves spatial size by adding zeros around the image. Stride is the step size "
    "of the filter. Larger stride reduces output dimensions, while padding maintains them.",

    "Q3. Receptive Field":
    "The region of the input image that influences a single neuron in deeper layers. "
    "It is important because larger receptive fields allow deeper CNNs to capture global patterns.",

    "Q4. Filter Size & Stride vs Parameters":
    "Larger filters increase parameters and computation, while smaller filters reduce them. "
    "Stride controls overlap; larger stride reduces computation but may miss details.",

    "Q5. LeNet vs AlexNet vs VGG":
    "LeNet (1998): shallow (5 layers), small filters (5x5), for digit recognition.\n"
    "AlexNet (2012): deeper (8 layers), larger filters (11x11 initially), ReLU + dropout, won ImageNet.\n"
    "VGG (2014): very deep (16–19 layers), small filters (3x3), uniform architecture, strong performance."


In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets as torch_datasets, transforms
import matplotlib.pyplot as plt


In [4]:
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train = x_train.reshape(-1,28,28,1).astype("float32") / 255.0
x_test = x_test.reshape(-1,28,28,1).astype("float32") / 255.0

model_mnist = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_mnist.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

history = model_mnist.fit(x_train, y_train, epochs=3, validation_split=0.1, verbose=1)
test_loss, test_acc = model_mnist.evaluate(x_test, y_test, verbose=0)
print("MNIST Test Accuracy (Keras):", test_acc)

# --- Q7: CNN on CIFAR-10 (Keras) ---
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()
x_train, x_test = x_train/255.0, x_test/255.0

model_cifar = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_cifar.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

history = model_cifar.fit(x_train, y_train, epochs=3, validation_split=0.1, verbose=1)
test_loss, test_acc = model_cifar.evaluate(x_test, y_test, verbose=0)
print("CIFAR-10 Test Accuracy (Keras):", test_acc)

# --- Q8: CNN on MNIST (PyTorch) ---
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torch_datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torch_datasets.MNIST(root='./data', train=False, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        # The input size to the linear layer is calculated based on the output size of the last pooling layer.
        # Input size: 1x28x28
        # After conv1 (kernel 3x3, stride 1): 32x26x26 ( (28 - 3)/1 + 1 )
        # After pool (kernel 2x2, stride 2): 32x13x13 ( 26 / 2 )
        # After conv2 (kernel 3x3, stride 1): 64x11x11 ( (13 - 3)/1 + 1 )
        # After pool (kernel 2x2, stride 2): 64x5x5 ( 11 / 2 = 5.5, round down to 5 )
        self.fc1 = nn.Linear(64 * 5 * 5, 128) # Corrected input size to the linear layer
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return torch.log_softmax(x, dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_torch = CNN().to(device)
optimizer = optim.Adam(model_torch.parameters(), lr=0.001)

for epoch in range(1, 2):  # 1 epoch for demo
    model_torch.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model_torch(data)
        loss = nn.NLLLoss()(output, target)
        loss.backward()
        optimizer.step()
print("PyTorch MNIST training done (1 epoch).")

# --- Q9: Custom Dataset with Keras ImageDataGenerator ---
# (Here we simulate with CIFAR-10 directory structure for demo purpose)
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
# Normally: train_gen = datagen.flow_from_directory("path/to/data/train", ...)
print("ImageDataGenerator example created (ready for custom dataset).")

# --- Q10: End-to-End Chest X-Ray Web App (Concept) ---
print("""
End-to-End Approach:
1. Data Preparation: Collect chest X-ray dataset, preprocess (resize, normalize, augment).
2. Model Training: Use CNN (Keras) with Conv2D, MaxPooling, Dense layers. Train to classify Normal vs Pneumonia.
3. Evaluation: Evaluate on validation/test set.
4. Deployment: Save model (model.h5). Build Streamlit app:
   - Upload X-ray image
   - Preprocess image
   - Predict using trained model
   - Display result to user
5. Host app on cloud (Streamlit Cloud / Heroku).
""")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - accuracy: 0.8910 - loss: 0.3680 - val_accuracy: 0.9782 - val_loss: 0.0732
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - accuracy: 0.9793 - loss: 0.0690 - val_accuracy: 0.9865 - val_loss: 0.0544
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 39s 16ms/step - accuracy: 0.9867 - loss: 0.0431 - val_accuracy: 0.9865 - val_loss: 0.0612
MNIST Test Accuracy (Keras): 0.9824000000953674
Epoch 1/3
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 54s 38ms/step - accuracy: 0.3705 - loss: 1.7149 - val_accuracy: 0.5762 - val_loss: 1.2020
Epoch 2/3
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 52s 37ms/step - accuracy: 0.5852 - loss: 1.1718 - val_accuracy: 0.6442 - val_loss: 1.0313
Epoch 3/3
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.6484 - loss: 1.0076 - val_accuracy: 0.6684 - val_loss: 0.9948
CIFAR-10 Test Accuracy (Keras): 0.6480000019073486
PyTorch MNIST training done (1 epoch).
ImageDataGenerator example created (ready for custom dataset).

End-to-End A